# Preprocessing for the LFQ_diaPASEF dataset

In [1]:
import pandas as pd
import numpy as np
import re
from src.column_spec import ColumnSpec
from src.lfq_diaPASED_pretreatment import *
from src.transformations import *

from src.QC import *
from src.plotting_functions import *
pio.renderers.default = "png"

## Renaming and ordering columns

In [6]:
CELL_LINES = ["WT", "EGFRT693A", "BRAFS151A1", "SOS1S1178A", "SHOC2T71A", "BRAFS151A2", "GAB1Y259A", "RPS6KA3S375A"]
TIME_POINTS = ["full", "starve", "2", "5", "10", "15", "20", "30", "90"]
REPLICATES = ["r1", "r2", "r3"]
CONDITION = "EGF" #"_EGF_"
DATA_TYPE = "raw:abs"

# Cell lines naming dictionary
labes_dic = {}
c = 1
for cell in CELL_LINES:
    for tp in TIME_POINTS:
        for rep in REPLICATES:
            labes_dic[c] = cell + "_" + DATA_TYPE + "_" + CONDITION + "_" + tp + "_" + rep
            c += 1

# Control channels naming
MIX_LABELS = {"mix":  "MIX_" + DATA_TYPE + "_" + CONDITION + "_starve_r1",
              "mixb": "MIX_" + DATA_TYPE + "_" + CONDITION + "_starve_r2",
              "mixc": "MIX_" + DATA_TYPE + "_" + CONDITION + "_starve_r3",}


In [2]:
# Importing dataset
# df = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/dia-quant-output/abundance_multi-site_MS2quant_None.tsv", sep="\t")

# Annotation columns -> project names
regular_rename = {
    "Index": "peptide_index",
    "Gene": "protein_name",
    "ProteinID": "protein_Id",
    "Peptide": "peptide_seq",
}

# Run columns -> naming convention
rename_cols = {}
unmatched = []
for col in df.columns:
    if not col.startswith("E:"):
        continue
    token = re.search(r"_([^_]+)\.d$", col) #sample_number
    if token is None:
        unmatched.append(col)
        continue
    sample = token.group(1)
    if sample.isdigit() and int(sample) in labes_dic:
        rename_cols[col] = labes_dic[int(sample)]
    elif sample in MIX_LABELS:
        rename_cols[col] = MIX_LABELS[sample]
    else:
        unmatched.append(col)


df = df.rename(columns=rename_cols)
df = df.rename(columns=regular_rename)

# Standard site column
df["site"] = df["peptide_index"] + "~" + df["peptide_seq"]  # This peptide sequence is not the correct one (missing the localization in small letters)

# --- Column ordering, computed AFTER every rename ------------------------------------------
sample_cols = [name for name in labes_dic.values() if name in df.columns]
mix_cols = [name for name in MIX_LABELS.values() if name in df.columns]
run_cols = set(sample_cols) | set(mix_cols)
other_cols = [col for col in df.columns if col not in run_cols]

df = df[other_cols + sample_cols + mix_cols]

# --- Report --------------------------------------------------------------------------------
missing = [name for name in list(labes_dic.values()) + list(MIX_LABELS.values())
           if name not in df.columns]
print(f"renamed {len(sample_cols)} experimental runs + {len(mix_cols)} mix controls; "
      f"{len(other_cols)} annotation columns")
if missing:
    print(f"  expected but absent ({len(missing)}): {missing}")
if unmatched:
    print(f"  unrecognised run columns ({len(unmatched)}): {unmatched}")
assert len(other_cols) + len(sample_cols) + len(mix_cols) == df.shape[1], "columns lost while reordering"
assert not df.columns.duplicated().any(), "duplicate column names after renaming"

print(df.shape)
#df.to_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260821_peptide_MS2quant_None_renamed.tsv", sep="\t", index=False)
df.head(5)


renamed 216 experimental runs + 3 mix controls; 21 annotation columns
(71087, 240)


,peptide_index,protein_name,protein_Id,peptide_seq,SequenceWindow,Start,End,Peptide Length,Probability,Protein,...,RPS6KA3S375A_raw:abs_EGF_20_r3,RPS6KA3S375A_raw:abs_EGF_30_r1,RPS6KA3S375A_raw:abs_EGF_30_r2,RPS6KA3S375A_raw:abs_EGF_30_r3,RPS6KA3S375A_raw:abs_EGF_90_r1,RPS6KA3S375A_raw:abs_EGF_90_r2,RPS6KA3S375A_raw:abs_EGF_90_r3,MIX_raw:abs_EGF_starve_r1,MIX_raw:abs_EGF_starve_r2,MIX_raw:abs_EGF_starve_r3
0,P16333_147_176_1_1_S166,NCK1,P16333,GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK,KCSDGWWR.GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK.LAA...,146,178,33,1.0,P16333,...,1714.0734,NaN,3342.1428,2313.1060,NaN,NaN,1156.0444,2429.0979,3397.1465,2826.1235
1,Q9Y618_1595_1611_1_0,NCOR2,Q9Y618,EIAKSPHSTVPEHHPHPISPYEHLLR;SPHSTVPEHHPHPISPYEHLLR,RKLTSTPR.EIAKSPHSTVPEHHPHPISPYEHLLR.GVSGVDLY,1591,1616,26,1.0,Q9Y618,...,491.0169,1076.0327,463.0141,NaN,1074.0334,1567.0479,392.0133,NaN,942.0345,514.0154
2,Q9H2G2_565_571_2_2_S565T569,SLK,Q9H2G2,VDEDSAEDTQSNDGK;VDEDSAEDTQSNDGKEVVEVGQK,EAADVAQK.VDEDSAEDTQSNDGKEVVEVGQK.LINKPMVG,561,583,23,1.0,Q9H2G2,...,NaN,NaN,NaN,1756.0579,NaN,NaN,1320.0422,729.0223,800.0243,NaN
3,O15085_1452_1469_1_1_S1466,ARHGEF11,O15085,SLGGESSGGTTPVGSFHTEAAR,LAHRELLK.SLGGESSGGTTPVGSFHTEAAR.WTDGSLSP,1452,1473,22,1.0,O15085,...,801.0384,NaN,NaN,NaN,NaN,NaN,723.0341,NaN,1333.0662,NaN
4,P08670_412_420_1_1_S419,VIM,P08670,ISLPLPNFSSLNLR,LLEGEESR.ISLPLPNFSSLNLR.ETNLDSLP,411,424,14,1.0,P08670,...,367.0161,560.0273,784.0388,NaN,380.0154,545.0200,NaN,734.0343,497.0189,449.0199


### Incorprorate proper "site" identifier column

In [22]:
df = add_site_identificator(df,
                            mod_col = "Best Precursor for Quant",
                            peptide_index_col= "peptide_index")
print(df.shape)
df.head(5)

(71087, 240)


,peptide_index,protein_name,protein_Id,peptide_seq,SequenceWindow,Start,End,Peptide Length,Probability,Protein,...,RPS6KA3S375A_raw:abs_EGF_20_r3,RPS6KA3S375A_raw:abs_EGF_30_r1,RPS6KA3S375A_raw:abs_EGF_30_r2,RPS6KA3S375A_raw:abs_EGF_30_r3,RPS6KA3S375A_raw:abs_EGF_90_r1,RPS6KA3S375A_raw:abs_EGF_90_r2,RPS6KA3S375A_raw:abs_EGF_90_r3,MIX_raw:abs_EGF_starve_r1,MIX_raw:abs_EGF_starve_r2,MIX_raw:abs_EGF_starve_r3
0,P16333_147_176_1_1_S166,NCK1,P16333,GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK,KCSDGWWR.GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK.LAA...,146,178,33,1.0,P16333,...,2514.9519,NaN,2400.7507,3096.2546,NaN,NaN,1780.9912,3081.9907,4014.0857,2760.9353
1,Q9Y618_1595_1611_1_0,NCOR2,Q9Y618,EIAKSPHSTVPEHHPHPISPYEHLLR;SPHSTVPEHHPHPISPYEHLLR,RKLTSTPR.EIAKSPHSTVPEHHPHPISPYEHLLR.GVSGVDLY,1591,1616,26,1.0,Q9Y618,...,822.2573,1007.8937,431.0079,NaN,1303.1925,1395.4010,741.8533,NaN,949.6689,492.6543
2,Q9H2G2_565_571_2_2_S565T569,SLK,Q9H2G2,VDEDSAEDTQSNDGK;VDEDSAEDTQSNDGKEVVEVGQK,EAADVAQK.VDEDSAEDTQSNDGKEVVEVGQK.LINKPMVG,561,583,23,1.0,Q9H2G2,...,NaN,NaN,NaN,2697.4099,NaN,NaN,2382.4438,892.5172,841.7625,NaN
3,O15085_1452_1469_1_1_S1466,ARHGEF11,O15085,SLGGESSGGTTPVGSFHTEAAR,LAHRELLK.SLGGESSGGTTPVGSFHTEAAR.WTDGSLSP,1452,1473,22,1.0,O15085,...,1090.6652,NaN,NaN,NaN,NaN,NaN,1402.2428,NaN,1417.0917,NaN
4,P08670_412_420_1_1_S419,VIM,P08670,ISLPLPNFSSLNLR,LLEGEESR.ISLPLPNFSSLNLR.ETNLDSLP,411,424,14,1.0,P08670,...,436.2199,432.3813,649.3447,NaN,399.7882,408.1336,NaN,863.2152,532.2197,458.8945


### Saving dataset with only raw abundances

In [23]:
# df.to_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260819_peptide_MS2quant_Norm_preprocessed.tsv", sep= "\t", index=False)

# Data transformation

`run_diapasef_transformations` (`src/transformations.py`, *diaPASEF (LFQ) transformations* section)
applies the chain below to every cell line in one pass:

| step | columns produced |
|---|---|
| 1 | `raw:mean`, `raw:median`, `raw:sd`, `raw:cv` — zeros treated as missing, so a non-detection does not drag the mean down |
| 2 | `log2:abs` — per replicate, zeros → NaN (`log2(0)` is undefined, and a zero here means *not detected*) |
| 3 | `log2:mean`, `log2:median`, `log2:sd` — statistics *of the log2 values*, i.e. the log2 geometric mean |
| 4 | `log2:FC` — `log2:mean(t) − log2:mean(starve)`, per cell line per condition |
| 5 | `log2:scaled` — `log2:FC / max(|log2:FC|)`, the maximum taken per cell line **across its conditions**, so relative amplitude between conditions survives |
| 6 | `log2:zscore` — `(log2:FC − mean_t) / sd_t`, per cell line **per condition**, i.e. shape with the amplitude removed |

**No starve, no fold change.** A site with no abundance detected in starve has no reference, so the
site is skipped and all of its `log2:FC` columns for that cell line stay **NaN** — not 0, which would
read as "no change" everywhere downstream. This is decided per site *per cell line*: a site can have
a usable FC in WT and none in a mutant. The counts are printed so the loss is visible.
`log2:FC_{treatment}_starve` is computed and is identically 0 — the rest of the project relies on
that structural zero.

**Why a separate section rather than `run_all_transformations`.** Three things differ for this dataset:
`_sort_timepoints()` sorts against a fixed `_TP_ORDER` that has no 20 or 30 min, so the diaPASEF grid
would come out as `… 15, 90, 20, 30`; all 8 cell lines are processed together with each column assigned
by its exact first field (a name that prefixes another cannot pull in both); and the ~1500 new columns
are attached with one `concat` per block instead of column-by-column insertion. The TMT functions are
untouched.

**The normalisation basis, and why it is not the TMT one.** Steps 5 and 6 are two *alternative*
representations of the same `log2:FC` values — neither reads the other, both are written, the analysis
downstream picks one. They differ from `compute_scaled_fc()` / `compute_zscore_fc()` in a single
deliberate way: which timepoints define the normalisation.

- `full` is excluded from both bases (`exclude_from_scale`, `exclude_from_zscore_basis`). Full media is
  a different media state, not a response to EGF, and its |FC| vs starve is routinely the largest value
  in the row — as the scaling denominator it squashes the actual response, and in the z-score basis it
  was measured on hme1_2 to carry ~26% of the clustering variance (`clustering_method_decision.md` §1).
- `starve` is additionally excluded from the z-score basis: it is identically 0 in FC space, so it
  contributes a constant rather than information.

The columns are still written for **every** timepoint — only the basis changes. So `log2:scaled` at
`full` may exceed 1, and `log2:zscore` at `starve` reads as how far the baseline sits below the mean
response, in SDs. `min_zscore_timepoints=3` blanks sites whose basis has fewer than three measured
points, where a sd is not a shape. Passing `exclude_from_scale=()`, `exclude_from_zscore_basis=()` and
`min_zscore_timepoints=1` reproduces the TMT functions exactly (verified).

No p-values or FDR: differential statistics are computed downstream in R/limma.

In [4]:
# This cell is used to load the dataframe that was already treated by the limma batch correction script
df = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260824_peptide_MS2quant_None_renamed_batchcorrected_linear.tsv", sep="\t", low_memory=False)
df

,peptide_index,protein_name,protein_Id,peptide_seq,SequenceWindow,Start,End,Peptide Length,Probability,Protein,...,RPS6KA3S375A_raw:abs_EGF_20_r3,RPS6KA3S375A_raw:abs_EGF_30_r1,RPS6KA3S375A_raw:abs_EGF_30_r2,RPS6KA3S375A_raw:abs_EGF_30_r3,RPS6KA3S375A_raw:abs_EGF_90_r1,RPS6KA3S375A_raw:abs_EGF_90_r2,RPS6KA3S375A_raw:abs_EGF_90_r3,MIX_raw:abs_EGF_starve_r1,MIX_raw:abs_EGF_starve_r2,MIX_raw:abs_EGF_starve_r3
0,P16333_147_176_1_1_S166,NCK1,P16333,GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK,KCSDGWWR.GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK.LAA...,146,178,33,1.0000,P16333,...,2322.951155,NaN,2900.928205,3134.773724,NaN,NaN,1566.697596,3291.968578,2887.963066,2453.031442
1,Q9Y618_1595_1611_1_0,NCOR2,Q9Y618,EIAKSPHSTVPEHHPHPISPYEHLLR;SPHSTVPEHHPHPISPYEHLLR,RKLTSTPR.EIAKSPHSTVPEHHPHPISPYEHLLR.GVSGVDLY,1591,1616,26,1.0000,Q9Y618,...,597.883405,944.214315,433.340267,NaN,942.459937,1466.618308,477.332342,NaN,826.631439,481.072976
2,Q9H2G2_565_571_2_2_S565T569,SLK,Q9H2G2,VDEDSAEDTQSNDGK;VDEDSAEDTQSNDGKEVVEVGQK,EAADVAQK.VDEDSAEDTQSNDGKEVVEVGQK.LINKPMVG,561,583,23,1.0000,Q9H2G2,...,NaN,NaN,NaN,1727.592496,NaN,NaN,1298.644537,717.204971,852.608474,NaN
3,O15085_1452_1469_1_1_S1466,ARHGEF11,O15085,SLGGESSGGTTPVGSFHTEAAR,LAHRELLK.SLGGESSGGTTPVGSFHTEAAR.WTDGSLSP,1452,1473,22,1.0000,O15085,...,856.189991,NaN,NaN,NaN,NaN,NaN,772.815086,NaN,1049.638015,NaN
4,P08670_412_420_1_1_S419,VIM,P08670,ISLPLPNFSSLNLR,LLEGEESR.ISLPLPNFSSLNLR.ETNLDSLP,411,424,14,1.0000,P08670,...,381.089083,566.180999,746.878696,NaN,384.191090,519.188371,NaN,762.180346,502.480249,427.738267
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71082,Q13112_507_538_1_1_S538,CHAF1B,Q13112,TDTPPSSVPTSVISTPSTEEIQSETPGDAQGSPPELKR;TDTPPSS...,RINLTPLK.TDTPPSSVPTSVISTPSTEEIQSETPGDAQGSPPELK...,507,543,37,1.0000,Q13112,...,78026.097279,104967.314546,102126.830700,84118.130349,94151.630060,88267.533835,68778.976756,98746.742986,116092.754337,89805.052068
71083,Q9H3Q1_9_14_1_1_S11,CDC42EP4,Q9H3Q1,QLVSSSVHSK,MPILK.QLVSSSVHSK.RRSRADLT,6,15,10,0.9997,Q9H3Q1,...,NaN,377.011200,NaN,NaN,NaN,NaN,393.154436,NaN,NaN,NaN
71084,Q13470_543_553_1_1_S543,TNK1,Q13470,AVPQGPPGLPPRPPLSSSSPQPSQPSR,PPEIRQAR.AVPQGPPGLPPRPPLSSSSPQPSQPSR.ERLPWPKR,528,554,27,0.9999,Q13470,...,2098.052386,2306.375887,2667.973164,1214.201067,1816.229884,2248.658176,NaN,4657.936188,3502.123878,4226.734075
71085,Q9UQ35_332_335_1_1_Y335,SRRM2,Q9UQ35,QPSSPYEDKDKDK;QPSSPYEDKDK,SSPETATK.QPSSPYEDKDKDK.KEKSATRP,330,342,13,1.0000,Q9UQ35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# All 8 cell lines in one pass. min_reps=1 reports whatever was detected; raise it to 2 to blank statistics computed from a single replicate (sd is NaN for a single replicate either way).
df_transformed = run_diapasef_transformations(df,
                                              cell_lines=CELL_LINES,
                                              conditions=["_EGF_"],
                                              data_type= DATA_TYPE,
                                              min_reps=1,
                                              reference="starve",
                                              verbose=True,)
print(df_transformed.shape)
df_transformed.head()

Found 8 cell lines, 72 (cell line, condition, timepoint) groups.
Sites without a 'starve' reference (log2:FC left as NaN):
  WT             EGF          20837 / 71087 (29.3%)
  EGFRT693A      EGF          22796 / 71087 (32.1%)
  BRAFS151A1     EGF          21846 / 71087 (30.7%)
  SOS1S1178A     EGF          22546 / 71087 (31.7%)
  SHOC2T71A      EGF          27620 / 71087 (38.9%)
  BRAFS151A2     EGF          22259 / 71087 (31.3%)
  GAB1Y259A      EGF          20523 / 71087 (28.9%)
  RPS6KA3S375A   EGF          23672 / 71087 (33.3%)
log2:scaled — scale factor from timepoints excluding ('full',):
  WT               49221 / 71087 sites scaled
  EGFRT693A        47516 / 71087 sites scaled
  BRAFS151A1       48148 / 71087 sites scaled
  SOS1S1178A       48068 / 71087 sites scaled
  SHOC2T71A        43144 / 71087 sites scaled
  BRAFS151A2       48210 / 71087 sites scaled
  GAB1Y259A        50006 / 71087 sites scaled
  RPS6KA3S375A     46881 / 71087 sites scaled
log2:zscore — basis excludes 

,peptide_index,protein_name,protein_Id,peptide_seq,SequenceWindow,Start,End,Peptide Length,Probability,Protein,...,GAB1Y259A_log2:zscore_EGF_90,RPS6KA3S375A_log2:zscore_EGF_full,RPS6KA3S375A_log2:zscore_EGF_starve,RPS6KA3S375A_log2:zscore_EGF_2,RPS6KA3S375A_log2:zscore_EGF_5,RPS6KA3S375A_log2:zscore_EGF_10,RPS6KA3S375A_log2:zscore_EGF_15,RPS6KA3S375A_log2:zscore_EGF_20,RPS6KA3S375A_log2:zscore_EGF_30,RPS6KA3S375A_log2:zscore_EGF_90
0,P16333_147_176_1_1_S166,NCK1,P16333,GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK,KCSDGWWR.GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK.LAA...,146,178,33,1.0,P16333,...,-2.318568,0.978401,2.180022,-0.098678,0.691042,0.461792,0.166701,0.750876,0.385994,-2.357727
1,Q9Y618_1595_1611_1_0,NCOR2,Q9Y618,EIAKSPHSTVPEHHPHPISPYEHLLR;SPHSTVPEHHPHPISPYEHLLR,RKLTSTPR.EIAKSPHSTVPEHHPHPISPYEHLLR.GVSGVDLY,1591,1616,26,1.0,Q9Y618,...,NaN,1.583711,0.349203,-0.274154,0.137486,0.322072,2.091572,-0.486044,-1.437488,-0.353443
2,Q9H2G2_565_571_2_2_S565T569,SLK,Q9H2G2,VDEDSAEDTQSNDGK;VDEDSAEDTQSNDGKEVVEVGQK,EAADVAQK.VDEDSAEDTQSNDGKEVVEVGQK.LINKPMVG,561,583,23,1.0,Q9H2G2,...,-0.178011,-2.026044,-0.499647,NaN,0.898802,-2.018516,-0.392530,0.483135,0.831346,0.197763
3,O15085_1452_1469_1_1_S1466,ARHGEF11,O15085,SLGGESSGGTTPVGSFHTEAAR,LAHRELLK.SLGGESSGGTTPVGSFHTEAAR.WTDGSLSP,1452,1473,22,1.0,O15085,...,-0.622165,1.182598,0.965639,0.925324,NaN,0.299982,-1.918422,0.604664,NaN,0.088452
4,P08670_412_420_1_1_S419,VIM,P08670,ISLPLPNFSSLNLR,LLEGEESR.ISLPLPNFSSLNLR.ETNLDSLP,411,424,14,1.0,P08670,...,-1.120215,1.317627,0.172879,-0.858616,0.696222,NaN,0.310656,-1.685073,1.343376,0.193435


### Checking the output how many sites are there for which a FC could be computed

If FC was computed this means that the site was detected at starvation and another time point (at least one)

In [6]:
site_row = df_transformed.index[0]

profile = {}
for dtype in ["raw:mean", "raw:median", "raw:sd", "raw:cv", "log2:mean", "log2:median", "log2:sd", "log2:FC", "log2:scaled", "log2:zscore"]:
    cols = ColumnSpec.select(df_transformed,
                             cell_lines=["WT"],
                             data_type=dtype,
                             conditions=["_" + CONDITION + "_"],)
    profile[dtype] = pd.Series(df_transformed.loc[site_row, cols].values, index=[c.split("_")[-1] for c in cols],)

print(df_transformed.loc[site_row, "site"])
pd.DataFrame(profile).T.round(3)

P16333_147_176_1_1_S166~GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK


,full,starve,2,5,10,15,20,30,90
raw:mean,3828.45876,3870.432618,2939.623576,2760.734583,2153.613881,1317.131322,NaN,2420.646032,1181.476269
raw:median,3828.45876,3980.464392,2369.025474,3064.990472,1991.254131,1163.853503,NaN,2420.646032,1181.476269
raw:sd,457.187342,1718.322294,1160.161004,542.343493,809.15383,682.014142,NaN,214.828225,195.044654
raw:cv,11.941812,44.39613,39.466312,19.644898,37.571908,51.780269,NaN,8.87483,16.508555
log2:mean,11.897386,11.809355,11.452873,11.41045,11.005047,10.232159,NaN,11.23833,10.196478
log2:median,11.897386,11.958721,11.210078,11.581667,10.959462,10.184694,NaN,11.23833,10.196478
log2:sd,0.172695,0.710551,0.530727,0.303759,0.539488,0.755574,NaN,0.128205,0.239259
log2:FC,0.088031,0.0,-0.356482,-0.398905,-0.804308,-1.577195,NaN,-0.571025,-1.612877
log2:scaled,0.05458,0.0,-0.221022,-0.247325,-0.498679,-0.977877,NaN,-0.354041,-1.0
log2:zscore,1.870509,1.701594,1.017575,0.936173,0.158284,-1.324737,NaN,0.605908,-1.393203


In [7]:
# How many sites ended up with a usable fold-change profile, per cell line — the complement of the "no starve reference" report printed above.
for cell in CELL_LINES:
    fc_cols = ColumnSpec.select(df_transformed,
                                cell_lines=[cell],
                                data_type="log2:FC",
                                conditions=["_" + CONDITION + "_"],)
    n_any = int(df_transformed[fc_cols].notna().any(axis=1).sum())
    n_all = int(df_transformed[fc_cols].notna().all(axis=1).sum())
    print(f"{cell:<14} FC at >=1 timepoint: {n_any:>6} | at every timepoint: {n_all:>6} "
          f"| of {len(df_transformed)}")

WT             FC at >=1 timepoint:  50250 | at every timepoint:  23175 | of 71087
EGFRT693A      FC at >=1 timepoint:  48291 | at every timepoint:  22873 | of 71087
BRAFS151A1     FC at >=1 timepoint:  49241 | at every timepoint:  23317 | of 71087
SOS1S1178A     FC at >=1 timepoint:  48541 | at every timepoint:  24158 | of 71087
SHOC2T71A      FC at >=1 timepoint:  43467 | at every timepoint:  23555 | of 71087
BRAFS151A2     FC at >=1 timepoint:  48828 | at every timepoint:  24676 | of 71087
GAB1Y259A      FC at >=1 timepoint:  50564 | at every timepoint:  25092 | of 71087
RPS6KA3S375A   FC at >=1 timepoint:  47415 | at every timepoint:  23348 | of 71087


## Step size between consecutive timepoints (`log2:step`)

`log2:FC` compares every timepoint against the **same** reference (starve), so a site that rises
early and then stays up keeps a large FC at every later timepoint even though nothing more is
happening. `log2:step` asks the complementary question — *what changed during this interval?*

    log2:step(t_i) = log2:FC(t_i) − log2:FC(t_(i-1))

with starve as the first reference, e.g. `WT_log2:step_EGF_2 = WT_log2:FC_EGF_2 − WT_log2:FC_EGF_starve`
and `WT_log2:step_EGF_5 = WT_log2:FC_EGF_5 − WT_log2:FC_EGF_2`.

`dia_log2_step_size()` is the diaPASEF twin of the TMT `log2_step_size()` used in
`TMT_dataset_preprocessing.ipynb` — same definition, but all 8 cell lines in one pass, cell lines
matched on the exact first field (`BRAFS151A1` vs `BRAFS151A2`), and one `concat` instead of
column-by-column insertion.

- A step column is named after the timepoint it **arrives at**: `..._log2:step_EGF_5` is the change
  between 2 and 5 min. So there is no step column for `starve` itself, and none for `full` — a
  separate media control, not the timepoint preceding starve, left out of the chain entirely
  (`exclude_full=True`).
- Since `log2:FC_*_starve` is identically 0, the first stimulation timepoint reproduces its own
  `log2:FC`. The baseline column is subtracted explicitly rather than assumed to be 0.
- The steps telescope: summing all steps of a condition returns the last `log2:FC` of that condition
  (checked below).
- ⚠️ The intervals are **strongly unequal** — 2, 5, 10, 15, 20, 30, 90 min, so the last step spans
  60 minutes and the first spans 2. A step is an increment *per interval*, not a rate per minute.
  Divide by the interval width if a rate is what is wanted.

**Missing values matter more here than in TMT.** A step joins two timepoints, so one missing
`log2:FC` blanks **two** step columns (the step into that timepoint and the step out of it). With
DIA's per-run missingness that is common rather than exceptional — the report below counts, per cell
line, how many sites have a complete step series. The gap is *not* bridged by skipping the missing
timepoint: that would silently change the interval a step covers.

In [8]:

# All 8 cell lines in one pass. The chain starts at starve and 'full' is excluded, so the step
# columns are 2, 5, 10, 15, 20, 30, 90 — one per interval after the baseline.
df_transformed = dia_log2_step_size(df_transformed,
                                    cell_lines=CELL_LINES,
                                    conditions=[CONDITION],
                                    baseline="starve",
                                    exclude_full=True,)

_step_wt = ColumnSpec.select(df_transformed,
                             cell_lines=["WT"],
                             data_type="log2:step",
                             conditions=["_" + CONDITION + "_"],)
print("\nWT step columns:", _step_wt)

# Sanity check: the steps telescope back to the last fold change of the condition. Compared on the
# shared index — sites with a hole in the series have a NaN sum and are counted, not silently dropped.
_step_sum = df_transformed[_step_wt].sum(axis=1, skipna=False)
_fc_last = df_transformed["WT_log2:FC_" + CONDITION + "_90"]
_comparable = _step_sum.notna() & _fc_last.notna()
print(f"sum(steps) == log2:FC at 90 min: {np.allclose(_step_sum[_comparable], _fc_last[_comparable])} "
      f"({(~_comparable).sum()} of {len(df_transformed)} site(s) skipped: incomplete series)")

df_transformed[["WT_log2:FC_" + CONDITION + "_starve", "WT_log2:FC_" + CONDITION + "_2",
                "WT_log2:FC_" + CONDITION + "_5"] + _step_wt].head()

log2:step — 56 columns (baseline 'starve', full excluded):
  WT             EGF        7 steps | complete series in   23990 / 71087 sites
  EGFRT693A      EGF        7 steps | complete series in   24046 / 71087 sites
  BRAFS151A1     EGF        7 steps | complete series in   24459 / 71087 sites
  SOS1S1178A     EGF        7 steps | complete series in   25228 / 71087 sites
  SHOC2T71A      EGF        7 steps | complete series in   24631 / 71087 sites
  BRAFS151A2     EGF        7 steps | complete series in   25683 / 71087 sites
  GAB1Y259A      EGF        7 steps | complete series in   26352 / 71087 sites
  RPS6KA3S375A   EGF        7 steps | complete series in   24736 / 71087 sites

WT step columns: ['WT_log2:step_EGF_2', 'WT_log2:step_EGF_5', 'WT_log2:step_EGF_10', 'WT_log2:step_EGF_15', 'WT_log2:step_EGF_20', 'WT_log2:step_EGF_30', 'WT_log2:step_EGF_90']
sum(steps) == log2:FC at 90 min: True (47097 of 71087 site(s) skipped: incomplete series)


,WT_log2:FC_EGF_starve,WT_log2:FC_EGF_2,WT_log2:FC_EGF_5,WT_log2:step_EGF_2,WT_log2:step_EGF_5,WT_log2:step_EGF_10,WT_log2:step_EGF_15,WT_log2:step_EGF_20,WT_log2:step_EGF_30,WT_log2:step_EGF_90
0,0.0,-0.356482,-0.398905,-0.356482,-0.042423,-0.405403,-0.772888,NaN,NaN,-1.041852
1,0.0,-0.108930,-0.216868,-0.108930,-0.107938,-0.579582,NaN,NaN,-0.230967,-0.676672
2,0.0,-0.277346,NaN,-0.277346,NaN,NaN,0.659168,NaN,NaN,-0.656028
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0,0.935661,0.649787,0.935661,-0.285874,-0.683390,0.601942,-0.304124,0.242671,NaN


## Fold change against WT starve (`log2:wtFC`)

`log2:FC` compares every cell line with **its own** starve, so any difference in basal
phosphorylation between cell lines cancels out. `log2:wtFC` uses **one common baseline, the WT starve**:

    log2:wtFC(X, t) = log2:mean(X, t) − log2:mean(WT, starve)

| cell line | `log2:wtFC` at a stimulation timepoint | `log2:wtFC` at `starve` | `log2:wtFC` at `full` |
|---|---|---|---|
| WT | = `log2:FC` | 0 | = `log2:FC` at full |
| mutant X | `log2:FC(X, t)` + basal offset | basal offset = X starve − WT starve | X full − WT starve |

So a mutant whose site already sits higher at rest shows that offset at every timepoint, including
`full` and `starve`, which are written too.

- A site with **no WT starve** gets NaN in every cell line. A mutant site with **no own starve** (no
  `log2:FC`) still gets a `log2:wtFC` if WT starve was measured. The printed report counts both.
- ⚠ diaPASEF has no between-cell-line normalisation. The basal offset therefore contains loading,
  clone and batch differences as well as biology. Read a mutant's `log2:wtFC_starve` as
  *basal difference + technical offset*.
- `dia_compute_wt_fold_change()` in `src/transformations.py`. It reads the `log2:mean` columns, so it
  runs after `run_diapasef_transformations`. It does not change `log2:FC`, `log2:scaled`,
  `log2:zscore` or `log2:step`, which stay relative to each cell line's own starve.

In [ ]:
# All 8 cell lines against the WT starve, every timepoint (full and starve included).
df_transformed = dia_compute_wt_fold_change(df_transformed,
                                            cell_lines=CELL_LINES,
                                            conditions=[CONDITION],
                                            reference_cell_line="WT",
                                            reference="starve",
                                            verbose=True,)

# Sanity checks: WT log2:wtFC equals its log2:FC at every timepoint, and for a mutant
# log2:wtFC(t) = log2:FC(t) + log2:wtFC(starve) wherever the mutant has its own starve.
for _tp in ["full", "starve", "2", "90"]:
    _a = df_transformed[f"WT_log2:wtFC_{CONDITION}_{_tp}"]
    _b = df_transformed[f"WT_log2:FC_{CONDITION}_{_tp}"]
    print(f"WT wtFC == FC at {_tp:>6}: {np.allclose(_a, _b, equal_nan=True)}")

_mut = "BRAFS151A1"
_own = df_transformed[f"{_mut}_log2:mean_{CONDITION}_starve"].notna()
_lhs = df_transformed[f"{_mut}_log2:wtFC_{CONDITION}_5"][_own]
_rhs = (df_transformed[f"{_mut}_log2:FC_{CONDITION}_5"] + df_transformed[f"{_mut}_log2:wtFC_{CONDITION}_starve"])[_own]
print(f"{_mut}: wtFC(5) == FC(5) + wtFC(starve): {np.allclose(_lhs, _rhs, equal_nan=True)}")

# Basal offset of every cell line vs WT (median over sites) — also shows any global loading shift.
_starve_cols = [f"{cell}_log2:wtFC_{CONDITION}_starve" for cell in CELL_LINES]
df_transformed[_starve_cols].describe().T.round(3)

In [9]:
#df_transformed.to_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260819_peptide_MS2quant_Norm_preprocessed.tsv", sep= "\t", index=False)
# df_transformed.to_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260824_peptide_MS2quant_None_renamed_batchcorrected_linear_transformed.tsv", sep= "\t", index=False)

## Peak timing (`peak:FC`, `peak:step`)

Two per-site labels that compress a whole profile into the timepoint that matters most:

| column | question |
|---|---|
| `{cell_line}_peak:FC_{condition}` | at which timepoint is **\|log2:FC\|** largest — when is the site furthest from its starve baseline? |
| `{cell_line}_peak:step_{condition}` | at which timepoint is **\|log2:step\|** largest — during which interval did the site change most? |

- The comparison is on the **absolute** value, so a downregulated site peaks at its deepest point and
  the largest negative step counts as a peak. The direction is therefore *not* recoverable from these
  columns — read the corresponding `log2:FC` / `log2:step` value for the sign.
- A step column is named after the timepoint it arrives at, so `peak:step = "5"` means the biggest
  change happened between 2 and 5 min, **not** after 5 min.
- `full` and `starve` are excluded from `peak:FC` (`exclude_timepoints`): `full` is a media control
  rather than a response, and `log2:FC` at starve is identically 0. Step columns never contain either.
- The value stored is the timepoint **label as a string** (`'2'`, `'90'`), NaN when the whole profile
  is missing. Ties go to the earliest timepoint.
- ⚠️ A partially observed site is scored over the timepoints it **does** have. With DIA missingness
  that is a real caveat: `peak:FC = '90'` on a site measured only at 30 and 90 min is the peak of what
  was measured, not the peak of the profile. The `no data:` counts below say how many sites had
  nothing to score; use the coverage filters in `diaPASEF_QC.ipynb` before reading these columns
  biologically.

These are flat per-site annotations — they carry no timepoint field, so they do not follow the
`{CellLine}_{DataType}_{Treatment}_{TimePoint}` scheme and `ColumnSpec.select()` ignores them, the
same as the kinase-prediction columns.

In [17]:
# df_transformed = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_transformed_limma_phPlus.tsv", sep="\t", low_memory=False)
df_transformed = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260824_peptide_MS2quant_None_renamed_batchcorrected_linear_transformed.tsv", sep="\t", low_memory=False)
df_transformed.head(5)

,peptide_index,protein_name,protein_Id,peptide_seq,SequenceWindow,Start,End,Peptide Length,Probability,Protein,...,GAB1Y259A_log2:step_EGF_20,GAB1Y259A_log2:step_EGF_30,GAB1Y259A_log2:step_EGF_90,RPS6KA3S375A_log2:step_EGF_2,RPS6KA3S375A_log2:step_EGF_5,RPS6KA3S375A_log2:step_EGF_10,RPS6KA3S375A_log2:step_EGF_15,RPS6KA3S375A_log2:step_EGF_20,RPS6KA3S375A_log2:step_EGF_30,RPS6KA3S375A_log2:step_EGF_90
0,P16333_147_176_1_1_S166,NCK1,P16333,GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK,KCSDGWWR.GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK.LAA...,146,178,33,1.0,P16333,...,-0.461595,-0.088951,-1.147110,-0.784596,0.271915,-0.078935,-0.101605,0.201142,-0.125635,-0.944711
1,Q9Y618_1595_1611_1_0,NCOR2,Q9Y618,EIAKSPHSTVPEHHPHPISPYEHLLR;SPHSTVPEHHPHPISPYEHLLR,RKLTSTPR.EIAKSPHSTVPEHHPHPISPYEHLLR.GVSGVDLY,1591,1616,26,1.0,Q9Y618,...,NaN,NaN,NaN,-0.255682,0.168842,0.075711,0.725794,-1.057258,-0.390253,0.444642
2,Q9H2G2_565_571_2_2_S565T569,SLK,Q9H2G2,VDEDSAEDTQSNDGK;VDEDSAEDTQSNDGKEVVEVGQK,EAADVAQK.VDEDSAEDTQSNDGKEVVEVGQK.LINKPMVG,561,583,23,1.0,Q9H2G2,...,-0.115137,-0.363571,0.025079,NaN,NaN,-1.895922,1.056704,0.569082,0.226297,-0.411756
3,O15085_1452_1469_1_1_S1466,ARHGEF11,O15085,SLGGESSGGTTPVGSFHTEAAR,LAHRELLK.SLGGESSGGTTPVGSFHTEAAR.WTDGSLSP,1452,1473,22,1.0,O15085,...,NaN,NaN,NaN,-0.011543,NaN,NaN,-0.635199,0.722439,NaN,NaN
4,P08670_412_420_1_1_S419,VIM,P08670,ISLPLPNFSSLNLR,LLEGEESR.ISLPLPNFSSLNLR.ETNLDSLP,411,424,14,1.0,P08670,...,-1.297778,0.620872,-0.497977,-0.486198,0.732877,NaN,NaN,-0.940693,1.427468,-0.542028


In [18]:
# peak:FC from the log2:FC columns, peak:step from the log2:step columns created above.
df_transformed = add_peak_timepoints(df_transformed,
                                     cell_lines=CELL_LINES,
                                     conditions=[CONDITION],
                                     data_types=("log2:FC", "log2:step",),
                                     exclude_timepoints=("full", "starve",),)

# Read one site as a sentence: where it peaked, and where it moved fastest, per cell line.
_peak_cols = [f"{cell}_peak:{kind}_{CONDITION}" for cell in CELL_LINES for kind in ("FC", "step")]
df_transformed[["site"] + _peak_cols].head()

peak timepoints — 16 columns:
  WT_peak:FC_EGF                     2: 3698, 5: 8758, 10: 6584, 15: 9354, 20: 4453, 30: 4601, 90: 11773 | no data: 21866
  EGFRT693A_peak:FC_EGF              2: 4350, 5: 4843, 10: 5560, 15: 8333, 20: 7722, 30: 7059, 90: 9649 | no data: 23571
  BRAFS151A1_peak:FC_EGF             2: 7643, 5: 4479, 10: 6721, 15: 7251, 20: 7257, 30: 7864, 90: 6933 | no data: 22939
  SOS1S1178A_peak:FC_EGF             2: 4044, 5: 8713, 10: 10468, 15: 4255, 20: 4052, 30: 10140, 90: 6396 | no data: 23019
  SHOC2T71A_peak:FC_EGF              2: 4858, 5: 2454, 10: 4294, 15: 8269, 20: 4410, 30: 4918, 90: 13941 | no data: 27943
  BRAFS151A2_peak:FC_EGF             2: 5848, 5: 6315, 10: 4311, 15: 6798, 20: 4808, 30: 6389, 90: 13741 | no data: 22877
  GAB1Y259A_peak:FC_EGF              2: 5014, 5: 4312, 10: 5282, 15: 11945, 20: 4186, 30: 4984, 90: 14283 | no data: 21081
  RPS6KA3S375A_peak:FC_EGF           2: 6715, 5: 5321, 10: 4810, 15: 7765, 20: 6733, 30: 6554, 90: 8983 | no data: 2

,site,WT_peak:FC_EGF,WT_peak:step_EGF,EGFRT693A_peak:FC_EGF,EGFRT693A_peak:step_EGF,BRAFS151A1_peak:FC_EGF,BRAFS151A1_peak:step_EGF,SOS1S1178A_peak:FC_EGF,SOS1S1178A_peak:step_EGF,SHOC2T71A_peak:FC_EGF,SHOC2T71A_peak:step_EGF,BRAFS151A2_peak:FC_EGF,BRAFS151A2_peak:step_EGF,GAB1Y259A_peak:FC_EGF,GAB1Y259A_peak:step_EGF,RPS6KA3S375A_peak:FC_EGF,RPS6KA3S375A_peak:step_EGF
0,P16333_147_176_1_1_S166~GSYNGQVGWFPSNYVTEEGDSP...,90,90,90,90,15,20,30,30,30,30,30,10,90,90,90,90
1,Q9Y618_1595_1611_1_0~EIAKSPHSTVPEHHPHPISPYEHLL...,90,90,10,10,2,2,20,20,30,30,90,30,None,None,30,20
2,Q9H2G2_565_571_2_2_S565T569~VDEDSAEDTQSNDGK;VD...,10,15,90,30,10,10,30,30,90,90,30,30,2,2,10,10
3,O15085_1452_1469_1_1_S1466~SLGGESSGGTTPVGSFHTEAAR,None,None,None,None,2,2,None,None,90,90,None,None,10,15,15,20
4,P08670_412_420_1_1_S419~ISLPLPNFSSLNLR,2,2,30,90,20,20,10,2,10,30,15,2,20,20,20,30


In [19]:
# Spot-check on one site: the labelled timepoints must be the argmax of |log2:FC| and |log2:step|.
# `full` / `starve` are dropped here for the same reason add_peak_timepoints() excludes them, and
# .abs().idxmax() is used rather than numpy argmax — argmax would return a NaN position.
_row = df_transformed[df_transformed["WT_peak:FC_" + CONDITION].notna()].index[0]
for _kind, _dtype in [("FC", "log2:FC"), ("step", "log2:step")]:
    _cols = ColumnSpec.select(df_transformed,
                              cell_lines=["WT"],
                              data_type=_dtype,
                              conditions=["_" + CONDITION + "_"],
                              exclude_full=True,)
    _profile = df_transformed.loc[_row, _cols].astype(float)
    _profile.index = [c.split("_")[-1] for c in _cols]
    _profile = _profile.drop(labels=["full", "starve"], errors="ignore")
    print(f"{_dtype:<10} " + "  ".join(f"{tp}:{v:6.2f}" for tp, v in _profile.items()))
    print(f"{'':<10} largest |value| at "
          f"{_profile.abs().idxmax() if _profile.notna().any() else 'nothing measured'}"
          f" | peak:{_kind} column says {df_transformed.loc[_row, 'WT_peak:' + _kind + '_' + CONDITION]}")


log2:FC    2: -0.36  5: -0.40  10: -0.80  15: -1.58  20:   nan  30: -0.57  90: -1.61
           largest |value| at 90 | peak:FC column says 90
log2:step  2: -0.36  5: -0.04  10: -0.41  15: -0.77  20:   nan  30:   nan  90: -1.04
           largest |value| at 90 | peak:step column says 90


In [19]:
# Save. Never overwrite an existing file — the name carries the transformation step.
# df_transformed.to_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_transformed_nolimma.tsv",sep="\t", index=False)
# df_transformed.head(200).to_csv("../../data/20260818_peptide_MS2quant_None_transformed_nolimma_sample.tsv",sep="\t", index=False)
df_transformed.head(5)

# Import Limma processed data and merge it

In [25]:
# df_transformed = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_transformed_nolimma.tsv",sep="\t")
limma_pvalues_df = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260824_peptide_MS2quant_None_renamed_batchcorrected_linear_transformed_limma_pvalues.tsv", sep="\t")
limma_pvalues_df.head(5)

,site,WT_log2:limmaFC_EGF_full,WT_log2:limmaFC_EGF_2,WT_log2:limmaFC_EGF_5,WT_log2:limmaFC_EGF_10,WT_log2:limmaFC_EGF_15,WT_log2:limmaFC_EGF_20,WT_log2:limmaFC_EGF_30,WT_log2:limmaFC_EGF_90,WT_log2:pvalue_EGF_full,...,RPS6KA3S375A_log2:adjustedFDR_EGF_15,RPS6KA3S375A_log2:adjustedFDR_EGF_20,RPS6KA3S375A_log2:adjustedFDR_EGF_30,RPS6KA3S375A_log2:adjustedFDR_EGF_90,RPS6KA3S375A_log2:Fpvalue_EGF_omnibus,RPS6KA3S375A_log2:Fpvalue_ALL_omnibus,RPS6KA3S375A_log2:FFDR_EGF_omnibus,RPS6KA3S375A_log2:FFDR_ALL_omnibus,RPS6KA3S375A_log2:adjustedFFDR_EGF_omnibus,RPS6KA3S375A_log2:adjustedFFDR_ALL_omnibus
0,A0A0B4J2A2_93_116_1_0~HTGSGILSMANAGPNTNGSQFFICTAK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A0A0B4J2A2_93_119_1_0~HTGSGILSMANAGPNTNGSQFFIC...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.167229,0.147458,0.050094,0.054895,0.634012,0.634012,0.956148,0.956148,0.019475,0.019475
2,A0A3B3IU46_36_45_1_1_S36~RPPESPPIVEEWNSR,-0.649117,-0.234146,-0.417358,-1.750213,-0.638183,-0.704210,-0.091603,-1.167349,0.437534,...,0.002184,0.097839,0.001505,0.125557,0.903444,0.903444,0.989667,0.989667,0.004511,0.004511
3,A0AVK6_346_366_1_0~WTGPEISPNTSGSSPVIHFTPSDLEVR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0AVT1_951_953_1_1_S951~NGISFTIWDR,-0.031707,-1.026764,-0.400791,-1.094043,-0.344261,-0.582538,-1.114633,-1.420874,0.956395,...,0.164739,0.088103,0.024352,0.091330,0.785404,0.785404,0.975654,0.975654,0.010704,0.010704


In [20]:
hme1_diapasef_limma = merge_limma_results(df_transformed,
                                          limma_path = "../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_limma_pvalues.tsv",
                                          key="site_index")

  merged 304 limma columns from ../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_limma_pvalues.tsv
  0 / 71087 sites carry statistics (71087 not tested by limma)


/Users/ignacionavascamacho/PycharmProjects/TMT_Data_analysis/src/transformations.py:1923: UserWarning: merge_limma_results: 43823 site(s) in '../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_limma_pvalues.tsv' are absent from the dataset and were discarded by the left join.
  warnings.warn(


In [23]:
# hme1_diapasef_limma.to_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_transformed_limma.tsv", sep="\t", index=False)

# Merge Phosphositeplus data

In [21]:
functional_score_df = pd.read_csv("../../External_Data/Metadata/PhosphoSitePlus.tsv", sep="\t")
regulatory_sites = pd.read_csv("../../External_Data/Metadata/Phosphosite/Regulatory_sites.tsv", sep="\t")

print(f"functional_score_df columns: {functional_score_df.columns}")
print(f"regulatory_sites columns: {regulatory_sites.columns}")

functional_score_df columns: Index(['protein_Id', 'protein_name', 'prot_seq_position', 'aa', 'site',
       'functional_score', 'ms_lit', 'ERK_motif', 'ERK_ext_motif'],
      dtype='object')
regulatory_sites columns: Index(['GENE', 'protein_name', 'information', 'protein_Id', 'GENE_ID',
       'HU_CHR_LOC', 'ORGANISM', 'MOD_RSD', 'SITE_GRP_ID', 'SITE_+/-7_AA',
       'DOMAIN', 'ON_FUNCTION', 'ON_PROCESS', 'ON_PROT_INTERACT',
       'ON_OTHER_INTERACT', 'PMIDs', 'LT_LIT', 'MS_LIT', 'MS_CST', 'NOTES'],
      dtype='object')


In [22]:
df_with_extra_info = merge_functional_score(df = hme1_diapasef_limma,
                                            phosphosite_df = functional_score_df,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position")
df_with_extra_info = merge_phosphoplus_info(df = df_with_extra_info,
                                            phosphosite_df = functional_score_df,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position",
                                            adding_info = ["ERK_motif"])
df_with_extra_info = merge_phosphoplus_info(df = df_with_extra_info,
                                            phosphosite_df = regulatory_sites,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position",
                                            adding_info = ["ORGANISM", "ON_FUNCTION", "ON_PROCESS", "ON_PROT_INTERACT", 'ON_OTHER_INTERACT'],
                                            regulatory_sites= True)

In [24]:
#df_with_extra_info.to_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_transformed_limma_phPlus.tsv", sep="\t", index=False)
# df_with_extra_info.to_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260824_peptide_MS2quant_None_batchCorrected_transformed_limma_phPlus.tsv", sep="\t", index=False,)